# 01 — Data Validation

This notebook quantifies data coverage from the initial gathering phase *before* any modeling.
It answers five questions that determine whether we have enough usable data to proceed:

1. **Market population** — how many resolved markets, by type and season?
2. **Price coverage** — how many markets have a usable pre-game price snapshot?
3. **Match rate** — how well do moneyline markets join to NFL games?
4. **Calibration sanity** — does the Polymarket implied probability track realized outcomes?
5. **Feature overlap** — how much of the matched data has nflverse features available?

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from abcm import config

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)

print('Data dir:', config.DATA_DIR)
print('Years:', config.YEARS)

## Load raw data

In [ ]:
markets = pd.read_parquet(config.POLYMARKET_RAW_DIR / 'markets.parquet')
events = pd.read_parquet(config.POLYMARKET_RAW_DIR / 'events.parquet')
schedules = pd.read_parquet(config.NFLVERSE_RAW_DIR / 'schedules.parquet')
print(f'markets: {len(markets):,} rows')
print(f'events:  {len(events):,} rows')
print(f'schedules: {len(schedules):,} games')

## 1. Market population by type and season

In [ ]:
markets['year'] = markets['market_end'].astype(str).str[:4]
type_year = markets.pivot_table(index='year', columns='market_type',
                                values='market_id', aggfunc='count', fill_value=0)
type_year.loc['Total'] = type_year.sum()
type_year

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
type_counts = markets['market_type'].value_counts()
ax.barh(type_counts.index[::-1], type_counts.values[::-1])
ax.set_xlabel('Number of resolved markets')
ax.set_title('Resolved NFL markets by type')
plt.tight_layout();

## 2. Price-history coverage

For each market we check whether its `candles.parquet` exists and how many
trades/candles it contains. The price fetch may still be running, so coverage
grows over runs.

In [ ]:
from abcm.polymarket import prices

ml = markets[markets['market_type'] == 'moneyline'].copy()

def _coverage(row):
    c = prices.candles_path(str(row['condition_id']))
    t = prices.trades_path(str(row['condition_id']))
    has = c.exists()
    n_candles = len(prices.candles_for(str(row['condition_id']))) if has else 0
    return pd.Series({'has_candles': has, 'n_candles': n_candles})

cov = ml.sample(min(500, len(ml)), random_state=0).apply(_coverage, axis=1) if len(ml) else pd.DataFrame()
if len(cov):
    print(f'sample of {len(cov)} moneyline markets:')
    print(f'  with candles on disk: {cov["has_candles"].sum()} ({cov["has_candles"].mean()*100:.0f}%)')
    print(f'  median candles (where present): {cov.loc[cov.has_candles, "n_candles"].median()}')
    print(f'  zero-trade markets: {(cov.loc[cov.has_candles, "n_candles"] == 0).sum()}')

## 3. Match rate (moneyline → NFL game)

Run the reconciler and inspect how cleanly markets join to scheduled games.

In [ ]:
from abcm.reconcile import match as match_mod

matched = match_mod.reconcile(markets, schedules)
ml_matched = matched[matched['market_type'] == 'moneyline']
status = ml_matched['match_status'].value_counts()
print('Moneyline match status:')
print(status)
print(f'\nMatch rate: {status.get("matched",0) / len(ml_matched) * 100:.1f}%')

In [ ]:
# Spot-check unmatched moneylines to find systemic causes.
unmatched = ml_matched[ml_matched['match_status'] == 'teams_found_no_game']
unmatched = unmatched.assign(month=unmatched['market_end'].astype(str).str[:7])
print('Unmatched moneylines by month (teams resolved, no scheduled game in window):')
print(unmatched['month'].value_counts().sort_index())

## 4. Calibration sanity

Group matched markets into bins by their pre-game snapshot price (implied
probability) and compare to the realized win rate. A well-calibrated market
sits near the diagonal. This is a market check, not a model check.

In [ ]:
from abcm.reconcile import snapshot as snap_mod

# Attach snapshots, then filter to markets with a clean cutoff snapshot.
with_snap = snap_mod.add_price_snapshots(ml_matched, schedules)
clean = with_snap[
    (with_snap['match_status'] == 'matched')
    & (with_snap['snapshot_method'] == 'cutoff')
    & with_snap['snapshot_price'].notna()
    & with_snap['resolved_yes_price'].notna()
].copy()
print(f'markets usable for calibration: {len(clean)}')
if len(clean) >= 20:
    clean['prob_bin'] = pd.cut(clean['snapshot_price'], bins=np.linspace(0, 1, 11))
    cal = clean.groupby('prob_bin', observed=True).agg(
        n=('resolved_yes_price', 'size'),
        realized=('resolved_yes_price', 'mean'),
        implied=('snapshot_price', 'mean'),
    )
    print(cal)
else:
    print('Not enough data with clean cutoff snapshots yet — run the price fetch longer.')

In [ ]:
if len(clean) >= 20:
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='perfect calibration')
    ax.scatter(cal['implied'], cal['realized'], s=cal['n'] * 3, alpha=0.7)
    ax.set_xlabel('Implied probability (snapshot price)')
    ax.set_ylabel('Realized win rate')
    ax.set_title('Polymarket calibration (market check)')
    ax.legend()
    plt.tight_layout();

## 5. Feature overlap with nflverse

Of the matched moneyline markets, how many reference games that have full
play-by-play data available (the prerequisite for EPA/CPOE feature engineering)?

In [ ]:
pbp = pd.read_parquet(config.NFLVERSE_RAW_DIR / 'pbp.parquet', columns=['game_id'])
pbp_games = set(pbp['game_id'].dropna().unique())

matched_games = matched[matched['game_id'].notna()]
matched_games = matched_games.assign(
    has_pbp=matched_games['game_id'].astype(str).isin(pbp_games)
)
print(f'matched markets with a game_id: {len(matched_games):,}')
print(f'  of which have pbp data: {matched_games["has_pbp"].sum():,} '
      f'({matched_games["has_pbp"].mean()*100:.1f}%)')

## Summary

Fill in the headline numbers once the price fetch completes:

| Metric | Value |
|---|---|
| Resolved NFL markets (total) | _ |
| Moneyline markets | _ |
| Moneylines matched to a game | _ |
| Moneylines with a clean pre-game snapshot | _ |
| Matched games with pbp coverage | _ |

**Decision gate:** if the clean-snapshot × matched count is in the low
hundreds or less, modeling will be high-variance and we should lean on the
binary-outcome target trained on nflverse history. If it's in the thousands,
a direct price-prediction / CLV approach is viable.